In [31]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
import optuna
from sklearn.svm import SVC
from sklearn.metrics import recall_score, f1_score, roc_auc_score


DATA_PATH = os.path.join('..', 'vr_legibility_train.csv')

In [44]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv(DATA_PATH)[["Yaw", "Pitch", "FontSize", "Label"]]
print(df)

# 특성(X)과 타겟(y) 분리
X = df.drop(['Label'] , axis=1)
y = df['Label']

# 학습/테스트 데이터 분할 9:1 (Stratified Sampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# 학습/테스트 데이터 분할 (StratifiedKFold 적용)
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

        Yaw  Pitch  FontSize  Label
0    -37.97  38.91     122.9      1
1     19.42  29.49     201.0      0
2     29.65  28.99       6.5      1
3    -39.31 -14.70      75.3      0
4     15.48  -6.88      75.3      0
...     ...    ...       ...    ...
4864  17.23 -22.17     201.0      0
4865  41.81 -12.94      10.6      1
4866 -50.93 -38.99     122.9      1
4867  17.37 -47.86     122.9      1
4868  35.24 -41.30      75.3      1

[4869 rows x 4 columns]


In [45]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
preprocessor = make_column_transformer(
    (MinMaxScaler(), ['Yaw', 'Pitch']),
    (make_pipeline(FunctionTransformer(np.log1p), MinMaxScaler()), ['FontSize'])
)

# 최종 파이프라인 (기존 Optuna의 'svm__C' 파라미터 이름을 그대로 쓰기 위해 Pipeline 유지)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', SVC(class_weight='balanced', random_state=42))
])

In [41]:
# 3. 교차 검증 객체 생성 및 Optuna를 활용한 하이퍼파라미터 탐색 레인지 설정
def objective(trial, X_train, y_train, pipeline):
    kernel = trial.suggest_categorical('svm__kernel', ['rbf', 'linear', 'poly'])
   
    params = {'svm__kernel': kernel}
        
    if kernel == 'rbf':
        params['svm__C'] = trial.suggest_float('svm__C', 0.1, 1000.0, log=True)
        params['svm__gamma'] = trial.suggest_float('svm__gamma', 1e-3, 10.0, log=True)
            
    elif kernel == 'linear':
        params['svm__C'] = trial.suggest_float('svm__C', 1e-3, 10.0, log=True)
            
    elif kernel == 'poly':
        params['svm__C'] = trial.suggest_float('svm__C', 0.1, 1.0, log=True)
        params['svm__gamma'] = trial.suggest_float('svm__gamma', 1e-3, 1.0, log=True)
        params['svm__degree'] = trial.suggest_int('svm__degree', 2, 3)
            
    # 파이프라인에 파라미터 적용
    pipeline.set_params(**params)
        
    scores = cross_val_score(
    pipeline, X_train, y_train, 
    cv=inner_cv, scoring='f1', n_jobs=-1,
    verbose=0
    )
    return scores.mean()

In [42]:
# 4. Outer CV 루프에서 Optuna 최적화 실행
import warnings
warnings.filterwarnings('ignore') # 경고 메시지 무시
optuna.logging.set_verbosity(optuna.logging.WARNING) # 로그가 너무 많이 뜨는 것을 방지 

f1_scores = []
rec_scores = []
auc_scores = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_train, y_train)):
    print(f"\n--- Outer Fold {fold_idx + 1} / 10 ---")
    X_train_val, X_test_val = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_train_val, y_test_val = y_train.iloc[train_idx], y_train.iloc[test_idx]

    # Optuna Study 생성 및 최적화 실행
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, X_train_val, y_train_val, pipeline), n_trials=120, n_jobs=1) 

    best_params_inner = study.best_params
    best_score_inner = study.best_value
    
    print("-" * 40)
    print(f"Best Hyper Params: {best_params_inner}")
    print(f"Best Inner CV F1:  {best_score_inner:.4f}")

    # 하이퍼파라미터로 모델 학습
    best_model_inner = pipeline.set_params(**best_params_inner)
    best_model_inner.fit(X_train_val, y_train_val)

    # 검증 데이터 평가
    y_pred = best_model_inner.predict(X_test_val)
    y_score = best_model_inner.decision_function(X_test_val)

    f1 = f1_score(y_test_val, y_pred)
    acc = recall_score(y_test_val, y_pred)
    roc_auc = roc_auc_score(y_test_val, y_score)

    #최고성능 하이퍼파라미터로 갱신
    if fold_idx == 0:
        best_param=best_params_inner
        best_f1=f1
    elif f1 > best_f1:
        best_param=best_params_inner
        best_f1=f1

    f1_scores.append(f1)
    rec_scores.append(acc)
    auc_scores.append(roc_auc)
    print(f"\n★ Outer fold Score:: F1 Score: {f1:.4f}, Recall: {acc:.4f}, ROC AUC: {roc_auc:.4f}")

print("\n"+"-" * 10 + " FINAL RESULTS " + "-" * 10)
print(f"최종 10-Fold 평균 F1 Score: {np.mean(f1_scores):.4f} (± {np.std(f1_scores):.4f})")
print(f"최종 10-Fold 평균 Recall: {np.mean(rec_scores):.4f} (± {np.std(rec_scores):.4f})")
print(f"최종 10-Fold 평균 ROC AUC: {np.mean(auc_scores):.4f} (± {np.std(auc_scores):.4f})")
print(f"최고 F1 하이퍼파라미터: {best_param}")


--- Outer Fold 1 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 7.524793959215649, 'svm__gamma': 0.47843398439210505}
Best Inner CV F1:  0.8195

★ Outer fold Score:: F1 Score: 0.7863, Recall: 0.7797, ROC AUC: 0.8737

--- Outer Fold 2 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 31.739591080669825, 'svm__gamma': 0.14828563659634775}
Best Inner CV F1:  0.8158

★ Outer fold Score:: F1 Score: 0.8291, Recall: 0.8220, ROC AUC: 0.8988

--- Outer Fold 3 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 666.8922100278243, 'svm__gamma': 0.11562115756979817}
Best Inner CV F1:  0.8184

★ Outer fold Score:: F1 Score: 0.8142, Recall: 0.8263, ROC AUC: 0.8913

--- Outer Fold 4 / 10 ---
----------------------------------------
Best Hyper Params: {'svm__kernel': 'rbf', 'svm__C': 159.9988123340056, 'svm__gamma': 0.18239984099189568}
Best Inne

In [43]:
# 5. 최종 모델 학습 및 저장
import joblib

print(" [최종 모델 학습 시작]")
print(X.columns)
print("=========================================")

#확률 옵션 키기
best_param['svm__probability'] = True

final_model = pipeline.set_params(**best_param)
final_model.fit(X_train, y_train) 

# 검증 데이터 평가
final_pred = final_model.predict(X_test)
final_roc_auc = final_model.decision_function(X_test)

final_f1 = f1_score(y_test, final_pred)
final_rec = recall_score(y_test, final_pred)
roc_auc = roc_auc_score(y_test, final_roc_auc)
print(f"최종 모델 F1 Score on Test Set: {final_f1:.4f}")
print(f"최종 모델 Recall on Test Set: {final_rec:.4f}")
print(f"최종 모델 AUC Score on Test Set: {roc_auc:.4f}")

# 학습이 완료된 final_model 객체를 하드디스크에 파일로 저장합니다.
joblib.dump(final_model, os.path.join('..', 'vr_legibility_svm_model.pkl'))
print("✅ 최종 모델이 'vr_legibility_svm_model.pkl' 파일로 저장되었습니다!")

 [최종 모델 학습 시작]
Index(['Yaw', 'FontSize'], dtype='object')
최종 모델 F1 Score on Test Set: 0.8136
최종 모델 Recall on Test Set: 0.8244
최종 모델 AUC Score on Test Set: 0.8893
✅ 최종 모델이 'vr_legibility_svm_model.pkl' 파일로 저장되었습니다!


In [ ]:
# 6. 모델 배포 후 예측 (새로운 데이터에 대한 예측)
import joblib
import pandas as pd
import os
from sklearn.metrics import recall_score, f1_score, roc_auc_score
# 1. 저장해둔 모델 불러오기 
deployed_model = joblib.load(os.path.join('..', 'vr_legibility_svm_model.pkl'))

# 2. 새 데이터 불러오기
"""
new_data = pd.DataFrame({
    'size': [180.0, 160.0],
    'yaw': [1250.5, -450.2],
    'pitch': [45.2, 210.8],
    "label" :[1, 1]
})
"""
new_data = pd.read_csv(os.path.join('..', 'coordinates.csv'))[["Yaw", "Pitch", 'FontSize', 'Label']]

X_new = new_data.drop('Label', axis=1)
y_true = new_data["Label"]

# 3. 새로운 데이터 예측
predictions = deployed_model.predict(X_new)
probabilities = deployed_model.predict_proba(X_new)[:, 1]
score = deployed_model.decision_function(X_new)

print("새로운 데이터 예측 완료! (총 {}개)".format(len(predictions)))
new_data['Prediction'] = predictions          # 0과 1 예측값 열 추가
new_data['Probability'] = probabilities       # 1일 확률 열 추가
print("\n--- 예측 결과 확인 ---")
print(new_data.head(10)) # 결과가 잘 들어갔는지 상위 10개 확인
new_data.to_csv('svm_new_data_predictions.csv', index=False) # 예측 결과를 CSV 파일로 저장
rec = recall_score(y_true, predictions)
f1 = f1_score(y_true, predictions)
auc = roc_auc_score(y_true, score)
print("\n" + "-" * 40)
print(f"새로운 데이터에 대한 Recall: {rec:.4f}")
print(f"새로운 데이터에 대한 F1 Score: {f1:.4f}")
print(f"새로운 데이터에 대한 ROC-AUC: {auc:.4f}")

새로운 데이터 예측 완료! (총 10767개)

--- 예측 결과 확인 ---
   Yaw  Pitch  Prediction  Probability
0  -55    -53           1          1.0
1  -55    -52           1          1.0
2  -55    -51           1          1.0
3  -55    -50           1          1.0
4  -55    -49           1          1.0
5  -55    -48           1          1.0
6  -55    -47           1          1.0
7  -55    -46           1          1.0
8  -55    -45           1          1.0
9  -55    -44           1          1.0


'\nrec = recall_score(y_true, predictions)\nf1 = f1_score(y_true, predictions)\nauc = roc_auc_score(y_true, score)\nprint("\n" + "-" * 40)\nprint(f"새로운 데이터에 대한 Recall: {rec:.4f}")\nprint(f"새로운 데이터에 대한 F1 Score: {f1:.4f}")\nprint(f"새로운 데이터에 대한 ROC-AUC: {auc:.4f}")\n'